In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [1]:
#Check imports
try:
  import torch
  import torch.nn as nn
  import pytorch_lightning as pl
  from torch.utils.data import DataLoader, random_split, Dataset
  import matplotlib.pyplot as plt
  import wandb
  from pytorch_lightning.loggers import WandbLogger
  from pytorch_lightning.callbacks import Callback
  from pytorch_msssim import ssim

except Exception as e:
  print(f"Exception = {e}")

In [3]:
import argparse
import datetime
import json
import math
import os
import sys
import time
import warnings
from functools import partial
from pathlib import Path
from typing import Dict, Iterable, List

import numpy as np
import torch
import torch.backends.cudnn as cudnn
import yaml

import utils
from multimae.criterion import MaskedMSELoss
from multimae.input_adapters import PatchedInputAdapter
from multimae.output_adapters import SpatialOutputAdapter
from utils import NativeScalerWithGradNormCount as NativeScaler
from utils import create_model
from utils.datasets_chloe import build_multimae_pretraining_dataset
from utils.optim_factory import create_optimizer
from utils.task_balancing import (NoWeightingStrategy,
                                  UncertaintyWeightingStrategy)

In [4]:
parser = argparse.ArgumentParser('MultiMAE pre-training script', add_help=False)
parser.add_argument('--batch_size', default=16, type=int,
                    help='Batch size per GPU (default: %(default)s)')
parser.add_argument('--epochs', default=100, type=int,
                    help='Number of epochs (default: %(default)s)')
parser.add_argument('--save_ckpt_freq', default=1, type=int,
                    help='Checkpoint saving frequency in epochs (default: %(default)s)')
# Task parameters
parser.add_argument('--in_domains', default='s1-s2', type=str,
                    help='Input domain names, separated by hyphen (default: %(default)s)')
parser.add_argument('--out_domains', default='cdl', type=str,
                    help='Output domain names, separated by hyphen (default: %(default)s)')

# Model parameters
parser.add_argument('--model', default='pretrain_multimae_base', type=str, metavar='MODEL',
                    help='Name of model to train (default: %(default)s)')
parser.add_argument('--num_encoded_tokens', default=784, type=int,
                    help='Number of tokens to randomly choose for encoder (default: %(default)s)')
parser.add_argument('--num_global_tokens', default=1, type=int,
                    help='Number of global tokens to add to encoder (default: %(default)s)')
parser.add_argument('--patch_size', default=16, type=int,
                    help='Base patch size for image-like modalities (default: %(default)s)')
parser.add_argument('--input_size', default=224, type=int,
                    help='Images input size for backbone (default: %(default)s)')
parser.add_argument('--alphas', type=float, default=0.3, 
                    help='Dirichlet alphas concentration parameter (default: %(default)s)')
parser.add_argument('--sample_tasks_uniformly', default=True, action='store_true',
                    help='Set to True/False to enable/disable uniform sampling over tasks to sample masks for.')
parser.add_argument('--decoder_use_task_queries', default=True, action='store_true',
                    help='Set to True/False to enable/disable adding of task-specific tokens to decoder query tokens')
parser.add_argument('--decoder_use_xattn', default=True, action='store_true',
                    help='Set to True/False to enable/disable decoder cross attention.')
parser.add_argument('--decoder_dim', default=256, type=int,
                    help='Token dimension inside the decoder layers (default: %(default)s)')
parser.add_argument('--decoder_depth', default=2, type=int,
                    help='Number of self-attention layers after the initial cross attention (default: %(default)s)')
parser.add_argument('--decoder_num_heads', default=8, type=int,
                    help='Number of attention heads in decoder (default: %(default)s)')
parser.add_argument('--drop_path', type=float, default=0.0, metavar='PCT',
                    help='Drop path rate (default: %(default)s)')
parser.add_argument('--loss_on_unmasked', default=False, action='store_true',
                    help='Set to True/False to enable/disable computing the loss on non-masked tokens')
parser.add_argument('--no_loss_on_unmasked', action='store_false', dest='loss_on_unmasked')
parser.set_defaults(loss_on_unmasked=False)
# Optimizer parameters
parser.add_argument('--opt', default='adamw', type=str, metavar='OPTIMIZER',
                    help='Optimizer (default: %(default)s)')
parser.add_argument('--opt_eps', default=1e-8, type=float, metavar='EPSILON',
                    help='Optimizer epsilon (default: %(default)s)')
parser.add_argument('--opt_betas', default=[0.9, 0.95], type=float, nargs='+', metavar='BETA',
                    help='Optimizer betas (default: %(default)s)')
parser.add_argument('--clip_grad', type=float, default=1, metavar='CLIPNORM',
                    help='Clip gradient norm (default: %(default)s)')
parser.add_argument('--skip_grad', type=float, default=None, metavar='SKIPNORM',
                    help='Skip update if gradient norm larger than threshold (default: %(default)s)')
parser.add_argument('--momentum', type=float, default=0.9, metavar='M',
                    help='SGD momentum (default: %(default)s)')
parser.add_argument('--weight_decay', type=float, default=0.05,
                    help='Weight decay (default: %(default)s)')
parser.add_argument('--weight_decay_end', type=float, default=None, help="""Final value of the
    weight decay. We use a cosine schedule for WD.  (Set the same value as args.weight_decay to keep weight decay unchanged)""")
parser.add_argument('--decoder_decay', type=float, default=None, help='decoder weight decay')
parser.add_argument('--blr', type=float, default=1e-5, metavar='LR',
                    help='Base learning rate: absolute_lr = base_lr * total_batch_size / 256 (default: %(default)s)')
parser.add_argument('--warmup_lr', type=float, default=1e-6, metavar='LR',
                    help='Warmup learning rate (default: %(default)s)')
parser.add_argument('--min_lr', type=float, default=0., metavar='LR',
                    help='Lower lr bound for cyclic schedulers that hit 0 (default: %(default)s)')
parser.add_argument('--task_balancer', type=str, default='none',
                    help='Task balancing scheme. One out of [uncertainty, none] (default: %(default)s)')
parser.add_argument('--balancer_lr_scale', type=float, default=1.0,
                    help='Task loss balancer LR scale (if used) (default: %(default)s)')
parser.add_argument('--warmup_epochs', type=int, default=0, metavar='N',
                    help='Epochs to warmup LR, if scheduler supports (default: %(default)s)')
parser.add_argument('--warmup_steps', type=int, default=-0, metavar='N',
                    help='Epochs to warmup LR, if scheduler supports (default: %(default)s)')
parser.add_argument('--fp32_output_adapters', type=str, default='',
                    help='Tasks output adapters to compute in fp32 mode, separated by hyphen.')
# Augmentation parameters
parser.add_argument('--hflip', type=float, default=0.5,
                    help='Probability of horizontal flip (default: %(default)s)')
parser.add_argument('--train_interpolation', type=str, default='bicubic',
                    help='Training interpolation (random, bilinear, bicubic) (default: %(default)s)')
# Dataset parameters
parser.add_argument('--data_path', type=str, default=None,
                    help='(optional) base dir if your txt paths are relative.')
parser.add_argument(
    '--s1_txt',
    type=str,
    default="/work/mech-ai-scratch/bgekim/project/imputation/MultiMAE_NEW/MultiMAE/valid_list/nova/30m/pair_S1.txt",
    help="Path to modis txt file"
)

parser.add_argument(
    '--s2_txt',
    type=str,
    default="/work/mech-ai-scratch/bgekim/project/imputation/MultiMAE_NEW/MultiMAE/valid_list/nova/30m/pair_S2.txt",
    help="Path to s2 txt file"
)

parser.add_argument(
    '--cdl_txt',
    type=str,
    default="/work/mech-ai-scratch/bgekim/project/imputation/MultiMAE_NEW/MultiMAE/valid_list/nova/30m/pair_CDL.txt",
    help="Path to CDL txt file"
)

parser.add_argument(
    '--all_domains',
    type=str,
    default='s1-s2-cdl',
    help='All domain names, separated by hyphen'
)


parser.add_argument('--imagenet_default_mean_and_std', default=False, action='store_true')
# Misc.
parser.add_argument('--output_dir', default='/work/mech-ai-scratch/bgekim/project/imputation/MultiMAE_NEW/MultiMAE/result/s1-s2-new/',
                    help='Path where to save, empty for no saving')
parser.add_argument('--device', default='cuda',
                    help='Device to use for training / testing')
parser.add_argument('--seed', default=0, type=int, help='Random seed ')
parser.add_argument('--resume', default='', help='resume from checkpoint')
parser.add_argument('--auto_resume', action='store_true')
parser.add_argument('--no_auto_resume', action='store_false', dest='auto_resume')
parser.set_defaults(auto_resume=True)
parser.add_argument('--start_epoch', default=0, type=int, metavar='N', help='start epoch')
parser.add_argument('--num_workers', default=4, type=int)
parser.add_argument('--pin_mem', action='store_true',
                    help='Pin CPU memory in DataLoader for more efficient (sometimes) transfer to GPU.')
parser.add_argument('--no_pin_mem', action='store_false', dest='pin_mem',
                    help='')
parser.set_defaults(pin_mem=False)
parser.add_argument('--find_unused_params', action='store_true')
parser.add_argument('--no_find_unused_params', action='store_false', dest='find_unused_params')
parser.set_defaults(find_unused_params=True)
# Wandb logging
parser.add_argument('--log_wandb', default=False, action='store_true',
                    help='Log training and validation metrics to wandb')
parser.add_argument('--no_log_wandb', action='store_false', dest='log_wandb')
parser.set_defaults(log_wandb=False)
parser.add_argument('--wandb_project', default='MultiMAE-RGB', type=str,
                    help='Project name on wandb')
parser.add_argument('--wandb_entity', default='goeulkim', type=str,
                    help='User or team name on wandb')
parser.add_argument('--wandb_run_name', default='multimae-modis-s2', type=str,
                    help='Run name on wandb')
parser.add_argument('--show_user_warnings', default=False, action='store_true')
# Distributed training parameters
parser.add_argument('--world_size', default=1, type=int,
                    help='number of distributed processes')
parser.add_argument('--local_rank', default=-1, type=int)
parser.add_argument('--dist_on_itp', action='store_true')
parser.add_argument('--dist_url', default='env://', help='url used to set up distributed training')


args = parser.parse_args(args=[])
args.all_domains = args.all_domains.split('-')

## Dataset Load

In [5]:
# Get dataset
dataset = build_multimae_pretraining_dataset(args)

In [6]:
print(len(dataset))

118059


In [7]:
# 2. 비율 정의 (예: 70% train, 15% val, 15% test)
train_ratio, val_ratio, test_ratio = 0.7, 0.15, 0.15
n_total = len(dataset)
n_train = int(n_total * train_ratio)
n_val = int(n_total * val_ratio)
n_test = n_total - n_train - n_val  # 나머지

# 3. split
train_dataset, val_dataset, test_dataset = random_split(
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(args.seed)  # reproducibility
)


print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))


82641
17708
17710


In [8]:
# 4. DataLoader 생성
train_loader = DataLoader(
    train_dataset,
    batch_size=args.batch_size,
    shuffle=True,
    num_workers=args.num_workers,
    pin_memory=args.pin_mem
)

val_loader = DataLoader(
    val_dataset,
    batch_size=args.batch_size,
    shuffle=False,
    num_workers=args.num_workers,
    pin_memory=args.pin_mem
)

test_loader = DataLoader(
    test_dataset,
    batch_size=args.batch_size,
    shuffle=False,
    num_workers=args.num_workers,
    pin_memory=args.pin_mem
)

/work/mech-ai-scratch/bgekim/miniconda3/envs/multimae_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 64 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [9]:
batch = next(iter(val_loader))
print(batch.keys())

dict_keys(['s1', 's2', 'cdl'])


In [10]:
for key, tensor in batch.items():
  print(f"{key}: {tensor.shape}")

s1: torch.Size([64, 2, 224, 224])
s2: torch.Size([64, 12, 224, 224])
cdl: torch.Size([64, 224, 224])


## From MultiMAE

## Pretraining

In [5]:
import torch
import torch.backends.cudnn as cudnn
from functools import partial
from multimae.criterion import MaskedMSELoss
from multimae.input_adapters import PatchedInputAdapter
from multimae.output_adapters import SpatialOutputAdapter
from utils import create_model, NativeScalerWithGradNormCount as NativeScaler
from utils.optim_factory import create_optimizer
import utils
import wandb

import torch
import torch.nn.functional as F

In [ ]:
# # add contrastive loss
# def contrastive_loss(z1, z2, temperature=0.1):
#     """
#     Cross-modal contrastive loss (NT-Xent style).
    
#     Args:
#         z1, z2: [B, D] latent vectors from two modalities (e.g., S1 and S2)
#         temperature: scaling factor for logits
    
#     Returns:
#         Scalar contrastive loss
#     """
#     # Normalize embeddings
#     z1 = F.normalize(z1, dim=1)
#     z2 = F.normalize(z2, dim=1)

#     # Compute similarity matrix
#     logits = torch.matmul(z1, z2.T) / temperature  # [B, B]
#     labels = torch.arange(z1.size(0), device=z1.device)

#     # Cross entropy loss for both directions
#     loss_12 = F.cross_entropy(logits, labels)
#     loss_21 = F.cross_entropy(logits.T, labels)

#     return (loss_12 + loss_21) / 2

In [6]:
# ------------------------------
# Input Domain Configuration (수정: 채널 수 맞게!)
# ------------------------------
DOMAIN_CONF = {
    's1': {
        'channels': 2,
        'stride_level': 1,
        'input_adapter': partial(PatchedInputAdapter, num_channels=2),
        'output_adapter': partial(SpatialOutputAdapter, num_channels=2),
        'loss': MaskedMSELoss, 
    },
    's2': {
        'channels': 12,
        'stride_level': 1,
        'input_adapter': partial(PatchedInputAdapter, num_channels=12),
        'output_adapter': partial(SpatialOutputAdapter, num_channels=12),
        'loss': MaskedMSELoss,  
    },
}

In [7]:
# ------------------------------
# Model Builder
# ------------------------------
def get_model(in_domains, out_domains, patch_size=4, decoder_dim=256):
    input_adapters = {
        d: DOMAIN_CONF[d]['input_adapter'](stride_level=1, patch_size_full=patch_size)
        for d in in_domains
    }
    output_adapters = {
        d: DOMAIN_CONF[d]['output_adapter'](
            stride_level=1,
            patch_size_full=patch_size,
            dim_tokens=decoder_dim,
            depth=2,
            num_heads=8,
            use_task_queries=True,
            task=d,
            context_tasks=in_domains,
            use_xattn=True
        )
        for d in out_domains
    }
    return create_model(
        "pretrain_multimae_base",
        input_adapters=input_adapters,
        output_adapters=output_adapters,
        num_global_tokens=1,
        drop_path_rate=0.0
    )

In [8]:
# ------------------------------
# Training Loop
# ------------------------------
def train_one_epoch(model, train_loader, tasks_loss_fn, optimizer, device, epoch, loss_scaler, in_domains, out_domains, split="train"):
    if split == "train":
        model.train()
    else:
        model.eval()

    metric_logger = utils.MetricLogger(delimiter="  ")
    header = f"Epoch: [{epoch}]"

    for step, batch in enumerate(metric_logger.log_every(train_loader, 10, header)):
        tasks_dict = {t: ten.to(device, non_blocking=True) for t, ten in batch.items()}
        input_dict = {t: tasks_dict[t] for t in in_domains if t in tasks_dict}

        with torch.cuda.amp.autocast():
            preds, masks = model(input_dict, num_encoded_tokens=args.num_encoded_tokens) # original 
            # preds, masks, latents = model(input_dict, num_encoded_tokens=args.num_encoded_tokens, return_encoded=True) # to add contrastive loss

            task_losses = {}
            for task in out_domains:
                target = tasks_dict[task]
            
                # --- Temporal dataset용 CDL target 수정 ---
                if task == "cdl" and target.ndim == 4:
                    target = target[:, 0, ...]  # [B, T, H, W] → [B, H, W]
                    
                task_losses[task] = tasks_loss_fn[task](preds[task].float(), target)

            loss_recon = sum(task_losses.values())

            # # Contrastive loss (예: modis vs s2)
            # z_s1 = latents['s1'].mean(dim=1)   # [B, D]
            # z_s2    = latents['s2'].mean(dim=1)      # [B, D]
            # loss_contrast = contrastive_loss(z_s1, z_s2, temperature=0.1)

            # loss = loss_recon + 0.1*loss_contrast
            loss = loss_recon
        
        if split == "train":
            optimizer.zero_grad()
            grad_norm = loss_scaler(loss, optimizer, parameters=model.parameters(), clip_grad=args.clip_grad)
            torch.cuda.synchronize()
        else:
            grad_norm = 0.0  # validation은 grad 없음


        metric_logger.update(loss=loss.item(), grad_norm=grad_norm)
        for task, l in task_losses.items():
            metric_logger.update(**{f'{task}_loss': l.item()})
        
        wandb.log({
            "epoch": epoch,
            "step": step,
            f"{split}_loss_total": loss.item(),
            f"{split}_grad_norm": grad_norm,
            **{f"{split}_{task}_loss": l.item() for task, l in task_losses.items()}
        })

    metric_logger.synchronize_between_processes()
    print("Averaged stats:", metric_logger)
    return metric_logger


In [12]:
# # ------------------------------
# # Test Loop
# # ------------------------------
# def test_one_epoch(model, test_loader, tasks_loss_fn, device, epoch, in_domains, out_domains):


#     # ----------------- W&B Init -----------------
#     wandb.init(
#         project="multimae-newdataset",
#         entity="goeulkim"
#         # id="gq9vrsal",   # 네가 알려줄 run id
#         # resume="allow",  # 기존 run에 추가
#     )

#     model.eval()
#     metric_logger = utils.MetricLogger(delimiter="  ")
#     header = f"Test: [Epoch {epoch}]"

#     with torch.no_grad():
#         for step, batch in enumerate(metric_logger.log_every(test_loader, 10, header)):
#             tasks_dict = {t: ten.to(device, non_blocking=True) for t, ten in batch.items()}
#             input_dict = {t: tasks_dict[t] for t in in_domains if t in tasks_dict}

#             with torch.cuda.amp.autocast():
#                 preds, masks = model(input_dict, num_encoded_tokens=args.num_encoded_tokens)

#                 task_losses = {}
#                 for task in out_domains:
#                     target = tasks_dict[task]
#                     task_losses[task] = tasks_loss_fn[task](preds[task].float(), target)

#                 loss = sum(task_losses.values())

#             # grad 없음
#             grad_norm = 0.0  

#             # 기록
#             metric_logger.update(loss=loss.item(), grad_norm=grad_norm)
#             for task, l in task_losses.items():
#                 metric_logger.update(**{f'{task}_loss': l.item()})

#             wandb.log({
#                 "epoch": epoch,
#                 "step": step,
#                 "test_loss_total": loss.item(),
#                 "test_grad_norm": grad_norm,
#                 **{f"test_{task}_loss": l.item() for task, l in task_losses.items()}
#             })

#     metric_logger.synchronize_between_processes()
#     print("Test stats:", metric_logger)
#     return metric_logger


from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

# ------------------------------
# Test Loop
# ------------------------------
def test_one_epoch(model, test_loader, tasks_loss_fn, device, epoch, in_domains, out_domains):

    # ----------------- W&B Init -----------------
    wandb.init(
        project="multimae-newdataset",
        entity="goeulkim",
        reinit=True  # 매번 새로운 run 시작 (필요하면 유지/병합 옵션으로 변경 가능)
    )

    model.eval()
    metric_logger = utils.MetricLogger(delimiter="  ")
    header = f"Test: [Epoch {epoch}]"

    all_preds, all_targets = [], []

    with torch.no_grad():
        for step, batch in enumerate(metric_logger.log_every(test_loader, 10, header)):
            tasks_dict = {t: ten.to(device, non_blocking=True) for t, ten in batch.items()}
            input_dict = {t: tasks_dict[t] for t in in_domains if t in tasks_dict}

            with torch.cuda.amp.autocast():
                preds, masks = model(input_dict, num_encoded_tokens=args.num_encoded_tokens)

                task_losses = {}
                for task in out_domains:
                    target = tasks_dict[task]
                    pred = preds[task].float()

                    # --- loss ---
                    task_losses[task] = tasks_loss_fn[task](pred, target)

                    # --- flatten prediction/target 저장 ---
                    all_preds.append(pred.detach().cpu().numpy().reshape(-1))
                    all_targets.append(target.detach().cpu().numpy().reshape(-1))

                loss = sum(task_losses.values())

            # grad 없음
            grad_norm = 0.0  

            # 기록
            metric_logger.update(loss=loss.item(), grad_norm=grad_norm)
            for task, l in task_losses.items():
                metric_logger.update(**{f'{task}_loss': l.item()})

            # Step별로는 loss만 기록
            wandb.log({
                "epoch": epoch,
                "step": step,
                "test_loss_total": loss.item(),
                **{f"test_{task}_loss": l.item() for task, l in task_losses.items()}
            })

    # ----------------- 전체 지표 계산 -----------------
    all_preds = np.concatenate(all_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    mse = mean_squared_error(all_targets, all_preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(all_targets, all_preds)
    r2 = r2_score(all_targets, all_preds)

    metric_logger.synchronize_between_processes()
    print("Test stats:", metric_logger)
    print(f"✅ Test Epoch {epoch} | "
          f"MSE: {mse:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")

    # Epoch 요약만 로깅
    wandb.log({
        "epoch": epoch,
        "test_mse": mse,
        "test_rmse": rmse,
        "test_mae": mae,
        "test_r2": r2,
    })

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        **{k: meter.global_avg for k, meter in metric_logger.meters.items()}
    }


In [ ]:
# ------------------------------
# Main
# ------------------------------
def main(train_loader, val_loader):
    cudnn.benchmark = True
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ----------------- Config -----------------
    batch_size = args.batch_size
    epochs = args.epochs
    lr = args.blr
    patch_size = args.patch_size
    decoder_dim = args.decoder_dim

    in_domains = ['s1', 's2']
    out_domains = ['s1', 's2']

    wandb.init(
        project="multimae-newdataset",
        name="chloe_code",
        entity="goeulkim",
        config={
            "epochs":epochs,
            "lr":lr,
            "batch_size":batch_size,
            "patch_size":patch_size,
            "decoder_dim":decoder_dim
        }
    )

    # ----------------- Model -----------------
    model = get_model(in_domains, out_domains, patch_size, decoder_dim).to(device)
    
    # optimizer = create_optimizer(lr, model)
    # optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=args.weight_decay)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        eps=args.opt_eps,
        betas=tuple(args.opt_betas),
        weight_decay=args.weight_decay
        )

    loss_scaler = NativeScaler()

    tasks_loss_fn = {
        d: DOMAIN_CONF[d]['loss'](patch_size=patch_size, stride=1)
        for d in out_domains
    }

    torch.autograd.set_detect_anomaly(True)
    best_val_loss = float("inf")

    # ----------------- Training -----------------
    for epoch in range(epochs):
        train_one_epoch(model, train_loader, tasks_loss_fn, optimizer, device, epoch, loss_scaler, in_domains, out_domains, split="train")
        # train_one_epoch(model, val_loader, tasks_loss_fn, optimizer, device, epoch, loss_scaler, in_domains, out_domains, split="valid")

        # Validate
        val_stats = train_one_epoch(model, val_loader, tasks_loss_fn, optimizer, device, epoch, loss_scaler, in_domains, out_domains, split="valid")

        # Validation loss 가져오기
        val_loss = val_stats.meters['loss'].global_avg



    # ----------------- Save -----------------
    torch.save(model.state_dict(), "pretrain_multimae.pth")
    # wandb.finish()
    print("✅ Pretrained model saved at pretrain_multimae.pth")


if __name__ == "__main__":
    # train_loader already here
    # from my_dataset import train_loader
    # main(train_loader)
    pass


In [ ]:
main(train_loader, val_loader)

wandb: Currently logged in as: goeul8604 (goeulkim) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/work/mech-ai-scratch/bgekim/miniconda3/envs/multimae_env/lib/python3.10/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/work/mech-ai-scratch/bgekim/project/imputation/MultiMAE_NEW/MultiMAE/utils/native_scaler.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self._scaler = torch.cuda.amp.GradScaler(enabled=enabled)
/tmp/ipykernel_1953419/983767835.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/work/mech-ai-scratch/bgekim/project/imputation/MultiMAE_NEW/MultiMAE/multimae/multimae.py:385: FutureWarning: `torch.cuda.amp.autocast(args...)` is depre

loss contains NaN: False
loss contains NaN: False
Epoch: [0]  [   0/2583]  eta: 10:58:03  loss: 3.7516 (3.7516)  grad_norm: 25.7891 (25.7891)  s1_loss: 2.3486 (2.3486)  s2_loss: 1.4030 (1.4030)  time: 15.2860  data: 4.0632  max mem: 10350
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
Epoch: [0]  [  10/2583]  eta: 1:19:23  loss: 1.4555 (2.0085)  grad_norm: 6.1470 (9.2881)  s1_loss: 0.5649 (1.0763)  s2_loss: 0.7832 (0.9322)  time: 1.8512  data: 0.4349  max mem: 11442
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
l

epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
step,▇▅█▂▂▆▇▂▂▁▁▃▆▇▂▄▄█▁▁▃▄▆▃▃▂▃▄▅▆▁▂▅▆▂▃▆▂▂▁
train_grad_norm,██▄▄▃▃▃▂▃▄▃▂▂▂▂▃▂▁▁▂▂▂▁▁▂▂▂▂▁▁▁▁▁▂▁▂▁▁▁▁
train_loss_total,█▃▂▂▂▂▂▂▁▂▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_s1_loss,█▆▅▃▃▃▅▃▃▂▂▂▂▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_s2_loss,██▇▄▃▄▃▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁
valid_grad_norm,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
valid_loss_total,▇▇██▅▃▄▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▂▁▂▁▁▁▁▁▁▁▁▁
valid_s1_loss,▇█▆▆▄▄▄▃▄▃▂▃▂▂▂▂▂▂▂▁▂▁▁▁▂▁▂▂▁▁▁▁▁▂▁▁▁▁▁▁
valid_s2_loss,█▅▅▅▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁
epoch,19


✅ Pretrained model saved at pretrain_multimae.pth


### pretraining - TEST

In [13]:
# ----------------- Config -----------------
batch_size = args.batch_size
epochs = args.epochs
lr = args.blr
patch_size = args.patch_size
decoder_dim = args.decoder_dim
in_domains = ['s1', 's2']
out_domains = ['s1', 's2']


# ----------------- Model ----------------- 
model = get_model(in_domains, out_domains, patch_size, decoder_dim).to(device)


# ----------------- Load checkpoint -----------------
ckpt_path = "/work/mech-ai-scratch/bgekim/project/imputation/MultiMAE_NEW/MultiMAE/model/best_model.pth"
checkpoint = torch.load(ckpt_path, map_location=device)

# 모델 가중치 로드
model.load_state_dict(checkpoint["model_state_dict"], strict=True)

# 몇 epoch에서 저장했는지 확인
start_epoch = checkpoint["epoch"]
print(f"✅ Loaded model from epoch {start_epoch}")



tasks_loss_fn = {
    d: DOMAIN_CONF[d]['loss'](patch_size=patch_size, stride=1)
    for d in out_domains
}


# ----------------- Run Test -----------------
test_stats = test_one_epoch(
    model, 
    test_loader, 
    tasks_loss_fn, 
    device, 
    epoch=-1,          # testing only
    in_domains=in_domains, 
    out_domains=out_domains
)

print("Test finished. Averaged stats:", test_stats)

/work/mech-ai-scratch/bgekim/miniconda3/envs/multimae_env/lib/python3.10/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


✅ Loaded model from epoch 20


wandb: Currently logged in as: goeul8604 (goeulkim) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


/work/mech-ai-scratch/bgekim/miniconda3/envs/multimae_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 64 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/tmp/ipykernel_1706592/2298727833.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/work/mech-ai-scratch/bgekim/project/imputation/MultiMAE_NEW/MultiMAE/multimae/multimae.py:385: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


loss contains NaN: False
loss contains NaN: False
Test: [Epoch -1]  [  0/277]  eta: 1:46:22  loss: 0.0359 (0.0359)  grad_norm: 0.0000 (0.0000)  s1_loss: 0.0116 (0.0116)  s2_loss: 0.0243 (0.0243)  time: 23.0413  data: 19.0248  max mem: 2965
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
loss contains NaN: False
Test: [Epoch -1]  [ 10/277]  eta: 0:10:31  loss: 0.0348 (0.0342)  grad_norm: 0.0000 (0.0000)  s1_loss: 0.0113 (0.0111)  s2_loss: 0.0233 (0.0231)  time: 2.3649  data: 1.7301  max mem: 3347
loss contains NaN: False
loss contains NaN: False
loss contains NaN: Fal

## Finetune for dosnstream - CDL prediction

In [5]:
import torch
import torch.backends.cudnn as cudnn
from functools import partial
from multimae.criterion import MaskedMSELoss, MaskedCrossEntropyLoss
from multimae.input_adapters import PatchedInputAdapter
from multimae.output_adapters import SpatialOutputAdapter, SegmenterMaskTransformerAdapter
from utils import create_model, NativeScalerWithGradNormCount as NativeScaler
import utils
import wandb
from utils.temporal_model_chloe import TemporalEncoderWrapper

In [6]:
from utils.dataset_temporal_chloe import MultiTaskTemporalImageFolder, DataAugmentationForMultiMAE

all_domains = ["s1", "s2", "cdl"]
input_size = 224
hflip = False

txt_paths = {
    "s1": "/work/mech-ai-scratch/bgekim/project/imputation/MultiMAE_NEW/MultiMAE/valid_list/nova/30m/pair_temporal_S1.txt",
    "s2": "/work/mech-ai-scratch/bgekim/project/imputation/MultiMAE_NEW/MultiMAE/valid_list/nova/30m/pair_temporal_S2.txt",
    "cdl": "/work/mech-ai-scratch/bgekim/project/imputation/MultiMAE_NEW/MultiMAE/valid_list/nova/30m/pair_temporal_CDL.txt",
    }

dataset = MultiTaskTemporalImageFolder(
    tasks=all_domains,
    txt_paths=txt_paths,
    transform=DataAugmentationForMultiMAE(input_size, hflip, all_domains),
    T=5,
    )

sample = dataset[0]
print({k: v.shape for k, v in sample.items()})

✅ Found 4434 patches with full 5-month series for all domains
{'s1': torch.Size([5, 2, 224, 224]), 's2': torch.Size([5, 12, 224, 224]), 'cdl': torch.Size([5, 224, 224])}


In [7]:
# 2. 비율 정의 (예: 70% train, 15% val, 15% test)
train_ratio, val_ratio, test_ratio = 0.7, 0.15, 0.15
n_total = len(dataset)
n_train = int(n_total * train_ratio)
n_val = int(n_total * val_ratio)
n_test = n_total - n_train - n_val  # 나머지

# 3. split
train_dataset, val_dataset, test_dataset = random_split(
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(args.seed)  # reproducibility
)


print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))


3103
665
666


In [8]:
# 4. DataLoader 생성
train_loader = DataLoader(
    train_dataset,
    batch_size=args.batch_size,
    shuffle=True,
    num_workers=args.num_workers,
    pin_memory=args.pin_mem
)

val_loader = DataLoader(
    val_dataset,
    batch_size=args.batch_size,
    shuffle=False,
    num_workers=args.num_workers,
    pin_memory=args.pin_mem
)

test_loader = DataLoader(
    test_dataset,
    batch_size=args.batch_size,
    shuffle=False,
    num_workers=args.num_workers,
    pin_memory=args.pin_mem
)

In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
from functools import partial
import numpy as np
from sklearn.metrics import confusion_matrix
import wandb

from multimae.criterion import MaskedMSELoss, MaskedCrossEntropyLoss
from multimae.input_adapters import PatchedInputAdapter
from multimae.output_adapters import SpatialOutputAdapter, SegmenterMaskTransformerAdapter
from utils import create_model, NativeScalerWithGradNormCount as NativeScaler
import utils

# ------------------------------
# 3D CNN Temporal Encoder
# ------------------------------
class Conv3DTemporalEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        
        self.conv3d_blocks = nn.Sequential(
            # [B, C, T, H, W]
            nn.Conv3d(input_dim, hidden_dim//2, kernel_size=(3,3,3), padding=(1,1,1)),
            nn.BatchNorm3d(hidden_dim//2),
            nn.ReLU(inplace=True),
            
            nn.Conv3d(hidden_dim//2, hidden_dim, kernel_size=(3,3,3), padding=(1,1,1)),
            nn.BatchNorm3d(hidden_dim),
            nn.ReLU(inplace=True),
        )
        
        # Temporal pooling
        self.temporal_pool = nn.AdaptiveAvgPool3d((1, None, None))
        
    def forward(self, x_seq):
        """
        Args:
            x_seq: [B, T, C, H, W]
        Returns:
            [B, hidden_dim, H, W]
        """
        B, T, C, H, W = x_seq.shape
        
        # Reshape: [B, C, T, H, W]
        x = x_seq.transpose(1, 2).contiguous()
        
        # 3D Conv
        x = self.conv3d_blocks(x)  # [B, hidden_dim, T, H, W]
        
        # Temporal pooling
        x = self.temporal_pool(x)  # [B, hidden_dim, 1, H, W]
        x = x.squeeze(2)  # [B, hidden_dim, H, W]
        
        return x


# ------------------------------
# Temporal CDL Classifier
# ------------------------------
class TemporalCDLClassifier(nn.Module):
    def __init__(self, pretrained_model, temporal_hidden_dim=256, num_classes=3, patch_size=16, input_size=224):
        super().__init__()
        
        self.pretrained_model = pretrained_model
        self.patch_size = patch_size
        self.input_size = input_size
        self.num_classes = num_classes
        
        # Freeze pretrained encoder (optional)
        for param in self.pretrained_model.parameters():
            param.requires_grad = False
        print("✅ Pretrained encoder frozen")
        
        # 3D CNN temporal encoder
        # Input: reconstruction outputs (s1: 2ch, s2: 12ch) = 14ch
        self.temporal_encoder = Conv3DTemporalEncoder(
            input_dim=14,  # s1(2) + s2(12)
            hidden_dim=temporal_hidden_dim
        )
        
        # Segmentation head (upsampling + classification)
        self.seg_head = nn.Sequential(
            # Upsample 1: 14x14 -> 28x28
            nn.Conv2d(temporal_hidden_dim, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            
            # Upsample 2: 28x28 -> 56x56
            nn.Conv2d(128, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            
            # Upsample 3: 56x56 -> 112x112
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            
            # Upsample 4: 112x112 -> 224x224
            nn.Conv2d(32, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            
            # Final classification
            nn.Conv2d(16, num_classes, 1)
        )
    
    def extract_features_single_timestep(self, input_dict, num_encoded_tokens):
        """
        단일 timestep에 대해 pretrained model로 feature 추출
        """
        with torch.no_grad():
            preds, masks = self.pretrained_model(input_dict, num_encoded_tokens=num_encoded_tokens)
        
        # Reconstruction outputs를 feature로 사용
        features = torch.cat([preds['s1'], preds['s2']], dim=1)  # [B, 14, 224, 224]
        
        return features
    
    def forward(self, batch_dict, num_encoded_tokens):
        """
        Args:
            batch_dict: {
                's1': [B, T, 2, H, W],
                's2': [B, T, 12, H, W],
                'cdl': [B, T, H, W] or [B, H, W]
            }
        Returns:
            [B, num_classes, H, W]
        """
        s1_temporal = batch_dict['s1']  # [B, T, 2, 224, 224]
        s2_temporal = batch_dict['s2']  # [B, T, 12, 224, 224]
        
        B, T, _, H, W = s1_temporal.shape
        
        # Step 1: Extract features for each timestep
        temporal_features = []
        for t in range(T):
            input_dict = {
                's1': s1_temporal[:, t],  # [B, 2, 224, 224]
                's2': s2_temporal[:, t]   # [B, 12, 224, 224]
            }
            feat_t = self.extract_features_single_timestep(input_dict, num_encoded_tokens)
            temporal_features.append(feat_t)
        
        # Stack: [B, T, 14, 224, 224]
        temporal_features = torch.stack(temporal_features, dim=1)
        
        # Step 2: 3D CNN temporal encoding
        temporal_encoded = self.temporal_encoder(temporal_features)  # [B, 256, 224, 224]
        
        # Step 3: Segmentation head
        logits = self.seg_head(temporal_encoded)  # [B, 3, 224, 224]
        
        return logits
    
    def unfreeze_encoder(self):
        """Fine-tuning을 위해 encoder unfreeze"""
        for param in self.pretrained_model.parameters():
            param.requires_grad = True
        print("✅ Encoder unfrozen for fine-tuning")


# ------------------------------
# Metrics
# ------------------------------
def compute_confusion_matrix(logits, target, num_classes=3):
    pred = logits.argmax(dim=1).cpu().numpy().ravel()
    target = target.cpu().numpy().ravel()
    cm = confusion_matrix(target, pred, labels=list(range(num_classes)))
    return cm

def pixel_accuracy_all(logits, target):
    pred = logits.argmax(dim=1)
    correct = (pred == target).sum().item()
    total = target.numel()
    return correct / max(1, total)

def mean_iou(logits, target, num_classes=3):
    pred = logits.argmax(dim=1).cpu().numpy()
    target = target.cpu().numpy()
    ious = []
    for c in range(num_classes):
        pred_mask = (pred == c)
        target_mask = (target == c)
        inter = np.logical_and(pred_mask, target_mask).sum()
        union = np.logical_or(pred_mask, target_mask).sum()
        if union > 0:
            ious.append(inter / union)
    return float(np.mean(ious)) if ious else 0.0

def per_class_accuracy(logits, target, num_classes=3):
    pred = logits.argmax(dim=1)
    accs = []
    for c in range(num_classes):
        mask = (target == c)
        n = mask.sum().item()
        if n > 0:
            correct = ((pred == c) & mask).sum().item()
            accs.append(correct / n)
        else:
            accs.append(float('nan'))
    return accs


# ------------------------------
# Training Loop
# ------------------------------
def train_one_epoch(model, loader, criterion, optimizer, device, epoch, loss_scaler, args, split="train"):
    model.train() if split == "train" else model.eval()
    metric_logger = utils.MetricLogger(delimiter="  ")
    header = f"{split.capitalize()} Epoch: [{epoch}]"
    
    total_acc, total_miou, n_batches = 0.0, 0.0, 0

    for step, batch in enumerate(metric_logger.log_every(loader, 10, header)):
        # Move to device
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        
        # Target: 첫 번째 timestep의 CDL 사용 (또는 마지막)
        target = batch['cdl']
        if target.ndim == 4:  # [B, T, H, W]
            target = target[:, 0]  # 첫 timestep
        target = target.long()

        with torch.cuda.amp.autocast():
            logits = model(batch, num_encoded_tokens=args.num_encoded_tokens)
            loss = criterion(logits, target)
            
            # Metrics
            acc = pixel_accuracy_all(logits, target)
            miou = mean_iou(logits, target, num_classes=3)
            total_acc += acc
            total_miou += miou
            n_batches += 1

        if split == "train":
            optimizer.zero_grad()
            grad_norm = loss_scaler(loss, optimizer, parameters=model.parameters(), clip_grad=args.clip_grad)
            torch.cuda.synchronize()
        else:
            grad_norm = 0.0

        metric_logger.update(loss=loss.item(), acc=acc, miou=miou, grad_norm=grad_norm)
        
        wandb.log({
            "epoch": epoch,
            "step": step,
            f"{split}_loss": loss.item(),
            f"{split}_acc": acc,
            f"{split}_miou": miou,
            f"{split}_grad_norm": grad_norm,
        })

    avg_acc = total_acc / max(1, n_batches)
    avg_miou = total_miou / max(1, n_batches)
    
    metric_logger.synchronize_between_processes()
    print(f"{split.capitalize()} - Loss: {metric_logger.meters['loss'].global_avg:.4f}, "
          f"Acc: {avg_acc:.4f}, mIoU: {avg_miou:.4f}")
    
    wandb.log({
        "epoch": epoch,
        f"{split}_loss_avg": metric_logger.meters['loss'].global_avg,
        f"{split}_acc_avg": avg_acc,
        f"{split}_miou_avg": avg_miou,
    })

    return {
        "loss": metric_logger.meters['loss'].global_avg,
        "acc": avg_acc,
        "miou": avg_miou
    }


# ------------------------------
# Test Loop
# ------------------------------
def test_one_epoch(model, test_loader, criterion, device, epoch, args, num_classes=3):
    model.eval()
    metric_logger = utils.MetricLogger(delimiter="  ")
    header = f"Test: [Epoch {epoch}]"

    total_acc, total_miou, n_batches = 0.0, 0.0, 0
    per_class_acc_sum = np.zeros(num_classes, dtype=np.float64)
    per_class_counts = np.zeros(num_classes, dtype=np.int64)
    cm_total = np.zeros((num_classes, num_classes), dtype=np.int64)

    with torch.no_grad():
        for step, batch in enumerate(metric_logger.log_every(test_loader, 10, header)):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            
            target = batch['cdl']
            if target.ndim == 4:
                target = target[:, 0]
            target = target.long()

            with torch.cuda.amp.autocast():
                logits = model(batch, num_encoded_tokens=args.num_encoded_tokens)
                loss = criterion(logits, target)
                
                # Metrics
                acc = pixel_accuracy_all(logits, target)
                miou = mean_iou(logits, target, num_classes=num_classes)
                accs = per_class_accuracy(logits, target, num_classes=num_classes)
                cm = compute_confusion_matrix(logits, target, num_classes=num_classes)

                total_acc += acc
                total_miou += miou
                cm_total += cm
                n_batches += 1

                for c in range(num_classes):
                    if not np.isnan(accs[c]):
                        per_class_acc_sum[c] += accs[c]
                        per_class_counts[c] += 1

            metric_logger.update(loss=loss.item(), acc=acc, miou=miou)

    avg_acc = total_acc / max(1, n_batches)
    avg_miou = total_miou / max(1, n_batches)
    per_class_avg = per_class_acc_sum / np.maximum(1, per_class_counts)
    
    class_names = ['other', 'corn', 'soybean']
    
    metric_logger.synchronize_between_processes()
    print(f"\n✅ Test - Loss: {metric_logger.meters['loss'].global_avg:.4f}, "
          f"Acc: {avg_acc:.4f}, mIoU: {avg_miou:.4f}")
    for c, name in enumerate(class_names):
        print(f"   {name} Acc: {per_class_avg[c]:.4f}")
    print("Confusion Matrix:\n", cm_total)

    log_dict = {
        "epoch": epoch,
        "test_loss": metric_logger.meters['loss'].global_avg,
        "test_acc": avg_acc,
        "test_miou": avg_miou,
    }
    for c, name in enumerate(class_names):
        log_dict[f"test_acc_{name}"] = per_class_avg[c]

    wandb.log(log_dict)
    wandb.log({"confusion_matrix": wandb.Table(
        data=cm_total.tolist(),
        columns=[f"Pred_{n}" for n in class_names],
        rows=[f"True_{n}" for n in class_names]
    )})

    return {
        "test_acc": avg_acc,
        "test_miou": avg_miou,
        "confusion_matrix": cm_total,
        **{f"test_acc_{class_names[c]}": per_class_avg[c] for c in range(num_classes)}
    }


# ------------------------------
# Domain Configuration
# ------------------------------
DOMAIN_CONF = {
    's1': {
        'channels': 2,
        'stride_level': 1,
        'input_adapter': partial(PatchedInputAdapter, num_channels=2),
        'output_adapter': partial(SpatialOutputAdapter, num_channels=2),
        'loss': MaskedMSELoss, 
    },
    's2': {
        'channels': 12,
        'stride_level': 1,
        'input_adapter': partial(PatchedInputAdapter, num_channels=12),
        'output_adapter': partial(SpatialOutputAdapter, num_channels=12),
        'loss': MaskedMSELoss,  
    },
}


# ------------------------------
# Load Pretrained Model
# ------------------------------
def load_pretrained_model(checkpoint_path, patch_size=16, decoder_dim=256):
    in_domains = ['s1', 's2']
    out_domains = ['s1', 's2']  # Reconstruction
    
    input_adapters = {
        d: DOMAIN_CONF[d]['input_adapter'](stride_level=1, patch_size_full=patch_size)
        for d in in_domains
    }
    
    output_adapters = {
        d: DOMAIN_CONF[d]['output_adapter'](
            stride_level=1,
            patch_size_full=patch_size,
            dim_tokens=decoder_dim,
            depth=2,
            num_heads=8,
            use_task_queries=True,
            task=d,
            context_tasks=in_domains,
            use_xattn=True
        )
        for d in out_domains
    }
    
    model = create_model(
        "pretrain_multimae_base",
        input_adapters=input_adapters,
        output_adapters=output_adapters,
        num_global_tokens=1,
        drop_path_rate=0.0
    )
    
    state_dict = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(state_dict, strict=False)
    
    return model


# ------------------------------
# Main
# ------------------------------
def main(train_loader, val_loader, test_loader):
    cudnn.benchmark = True
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Config
    args = type("Args", (), {})()
    args.batch_size = 8
    args.epochs = 50
    args.blr = 1e-4
    args.patch_size = 16
    args.decoder_dim = 256
    args.num_encoded_tokens = 196  # (224/16)^2
    args.clip_grad = 1.0

    # Wandb
    wandb.init(
        project="temporal-cdl-classification",
        name="3D-CNN-temporal",
        entity="goeulkim",
        config=vars(args)
    )

    # Load pretrained model
    print("Loading pretrained MultiMAE...")
    pretrained_model = load_pretrained_model(
        checkpoint_path="/work/mech-ai-scratch/bgekim/project/imputation/MultiMAE_NEW/MultiMAE/model/best_model.pth",
        patch_size=args.patch_size,
        decoder_dim=args.decoder_dim
    ).to(device)

    # Build temporal classifier
    model = TemporalCDLClassifier(
        pretrained_model=pretrained_model,
        temporal_hidden_dim=256,
        num_classes=3,
        patch_size=args.patch_size,
        input_size=224
    ).to(device)
    
    print(f"Total params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")

    # Optimizer & Loss
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=args.blr,
        weight_decay=0.01
    )
    loss_scaler = NativeScaler()
    criterion = nn.CrossEntropyLoss(ignore_index=255)

    # Training loop
    best_miou = 0.0
    for epoch in range(args.epochs):
        print(f"\n{'='*50}")
        print(f"Epoch {epoch+1}/{args.epochs}")
        print(f"{'='*50}")
        
        train_one_epoch(model, train_loader, criterion, optimizer, device, epoch, loss_scaler, args, split="train")
        val_stats = train_one_epoch(model, val_loader, criterion, optimizer, device, epoch, loss_scaler, args, split="val")
        
        # Save best
        if val_stats['miou'] > best_miou:
            best_miou = val_stats['miou']
            torch.save(model.state_dict(), "best_temporal_cdl.pth")
            print(f"✅ Best model saved! (mIoU: {best_miou:.4f})")
        
        # Fine-tuning phase
        if epoch == args.epochs // 2:
            print("\n🔥 Starting fine-tuning phase...")
            model.unfreeze_encoder()
            optimizer = torch.optim.AdamW(model.parameters(), lr=args.blr/10, weight_decay=0.01)

    # Test
    test_stats = test_one_epoch(model, test_loader, criterion, device, args.epochs, args, num_classes=3)
    print("✅ Test performance:", test_stats)

    wandb.finish()
    print("✅ Training completed!")


if __name__ == "__main__":
    # from my_dataset import train_loader, val_loader, test_loader
    # main(train_loader, val_loader, test_loader)
    pass

In [33]:
main(train_loader, val_loader, test_loader)

Loading pretrained MultiMAE...
✅ Pretrained encoder frozen
Total params: 95.59M
Trainable params: 1.33M

Epoch 1/50


/work/mech-ai-scratch/bgekim/project/imputation/MultiMAE_NEW/MultiMAE/utils/native_scaler.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self._scaler = torch.cuda.amp.GradScaler(enabled=enabled)
/tmp/ipykernel_1928814/2939146095.py:231: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/work/mech-ai-scratch/bgekim/project/imputation/MultiMAE_NEW/MultiMAE/multimae/multimae.py:385: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


RuntimeError: input and target batch or spatial sizes don't match: target [16, 224, 224], input [16, 3, 3584, 3584]